## Exploración inicial de los datos - Enfermedad del corazón (Heart Disease)

### By:
Ronaldo Duran (JRDT)

### Date:
2026-08-21

### Description:

Notebook de exploración general de los datos del proyecto de clasificación de enfermedad
cardíaca, correspondiente al
[issue #8 - Exploración inicial de datos](https://github.com/ronaldo-duran/Hearth-project/issues/8).

Partiendo del CSV RAW caracterizado en el
[issue #7](https://github.com/ronaldo-duran/Hearth-project/issues/7), aquí se:

1. Describe el esquema y las características generales de los datos.
2. **Unifica la representación de los valores nulos**: el ruido inyectado en el dataset
   (texto basura en columnas numéricas, números sin sentido en categóricas, espacios
   sobrantes) son nulos disfrazados y se convierten en `pd.NA`.
3. **Convierte cada columna a su tipo correcto** (entero, decimal, booleano, categórico,
   ordinal) para que cada columna tenga un tipo de dato uniforme.
4. **Almacena el dataset tipado en `.parquet`**, formato que sí preserva los dtypes.

**Alcance**: esta etapa corresponde a la capa `data/02_intermediate` de la convención de
capas del proyecto (`data/README.md`), definida como *"modelo de datos que se introduce
para tipar el modelo de datos RAW"*. Por eso **no se eliminan ni se deduplican filas**:
el resultado conserva las 3030 filas del RAW, solo que correctamente tipadas. La
deduplicación y el descarte de filas inválidas pertenecen a la capa `03_primary` y se
harán en la etapa de limpieza (issue #11), donde importa el orden respecto al
`train_test_split` para no producir *data leakage*.

## 📚 Import libraries

In [ ]:
# base libraries for data science
from pathlib import Path

import pandas as pd

## 💾 Load data

In [ ]:
# Raíz del proyecto: se busca hacia arriba la carpeta data/01_raw
current_dir = Path.cwd().resolve()
project_root = next(
    p for p in [current_dir, *current_dir.parents] if (p / "data" / "01_raw").exists()
)

RAW_DIR = project_root / "data" / "01_raw"
INTERMEDIATE_DIR = project_root / "data" / "02_intermediate"

corazon_raw = pd.read_csv(RAW_DIR / "corazon.csv")
print(f"Dimensiones: {corazon_raw.shape[0]} filas x {corazon_raw.shape[1]} columnas")
corazon_raw.head()

Dimensiones: 3030 filas x 14 columnas


,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
0,63,Male,typical,145,233,1.0,left ventricular hypertrophy,150,0,2.3,3,0.0,fixed,0
1,67,Male,asymptomatic,160,286,0.0,left ventricular hypertrophy,108,1,1.5,2,3.0,normal,1
2,67,Male,asymptomatic,120,229,0.0,left ventricular hypertrophy,129,1,2.6,2,2.0,reversable,1
3,37,Male,nonanginal,130,250,0.0,normal,187,0,3.5,3,0.0,normal,0
4,41,Female,nontypical,130,204,0.0,left ventricular hypertrophy,172,0,1.4,1,0.0,normal,0


## 📊 Descripción general de los datos

Primer vistazo al esquema tal como lo entrega el CSV.

In [ ]:
corazon_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         3000 non-null   str    
 1   sex         2969 non-null   str    
 2   chest_pain  2947 non-null   str    
 3   rest_bp     2949 non-null   str    
 4   chol        2945 non-null   str    
 5   fbs         2933 non-null   float64
 6   rest_ecg    2837 non-null   str    
 7   max_hr      2859 non-null   str    
 8   exang       2879 non-null   str    
 9   old_peak    2880 non-null   str    
 10  slope       2879 non-null   str    
 11  ca          2868 non-null   str    
 12  thal        2904 non-null   str    
 13  disease     2924 non-null   str    
dtypes: float64(1), str(13)
memory usage: 506.9 KB


In [ ]:
# Resumen por columna: tipo, nulos y cardinalidad
resumen_raw = pd.DataFrame(
    {
        "dtype": corazon_raw.dtypes.astype(str),
        "no_nulos": corazon_raw.notna().sum(),
        "nulos": corazon_raw.isna().sum(),
        "pct_nulos": (corazon_raw.isna().mean() * 100).round(2),
        "valores_unicos": corazon_raw.nunique(),
    }
)
resumen_raw

,dtype,no_nulos,nulos,pct_nulos,valores_unicos
age,str,3000,30,0.99,43
sex,str,2969,61,2.01,5
chest_pain,str,2947,83,2.74,7
rest_bp,str,2949,81,2.67,52
chol,str,2945,85,2.81,154
fbs,float64,2933,97,3.20,2
rest_ecg,str,2837,193,6.37,8
max_hr,str,2859,171,5.64,92
exang,str,2879,151,4.98,4
old_peak,str,2880,150,4.95,42


In [ ]:
# La descripción estadística es inservible: pandas trata casi todo como texto
corazon_raw.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,3000,43,58,187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sex,2969,5,Male,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chest_pain,2947,7,asymptomatic,1394,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rest_bp,2949,52,120,361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chol,2945,154,204,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fbs,2933.0,NaN,NaN,NaN,0.148312,0.35547,0.0,0.0,0.0,0.0,1.0
rest_ecg,2837,8,normal,1409,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max_hr,2859,92,162,103,NaN,NaN,NaN,NaN,NaN,NaN,NaN
exang,2879,4,0,1929,NaN,NaN,NaN,NaN,NaN,NaN,NaN
old_peak,2880,42,0.0,944,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Lectura del diagnóstico

Trece de las catorce columnas se leyeron como **texto** (`str`) y solo `fbs` como
`float64`. La causa es el ruido inyectado a propósito en el dataset: basta un único valor
como `fggfds` en `age` para que pandas no pueda inferir el tipo numérico de toda la
columna. La consecuencia práctica es que `describe()` no produce ninguna estadística
útil: informa `count`/`unique`/`top`/`freq` en lugar de media, desviación y cuartiles.

**Ninguna exploración estadística es posible antes de sanear los tipos**, y ese es
justamente el objetivo de este notebook.

### Esquema objetivo

A partir del diccionario de `data/01_raw/datos_corazon_Info.txt`, este es el tipo al que
debe llegar cada columna:

| Variable | Tipo destino | Dominio válido |
| --- | --- | --- |
| `age` | `Int16` (entero anulable) | años, rango plausible 0-120 |
| `sex` | `category` | `Female`, `Male` |
| `chest_pain` | `category` | `asymptomatic`, `nonanginal`, `nontypical`, `typical` |
| `rest_bp` | `Int16` | mm Hg, rango plausible 50-250 |
| `chol` | `Int16` | mg/dl, rango plausible 100-600 |
| `fbs` | `boolean` | glucosa en ayunas > 120 mg/dl |
| `rest_ecg` | `category` | `normal`, `ST-T wave abnormality`, `left ventricular hypertrophy` |
| `max_hr` | `Int16` | lpm, rango plausible 60-220 |
| `exang` | `boolean` | angina inducida por ejercicio |
| `old_peak` | `Float64` (decimal anulable) | depresión ST, rango plausible 0.0-10.0 |
| `slope` | `Int8` (**ordinal**) | 1 < 2 < 3 |
| `ca` | `Int8` (conteo) | 0, 1, 2, 3 vasos |
| `thal` | `category` | `fixed`, `normal`, `reversable` |
| `disease` | `boolean` (**target**) | 1 = enfermedad, 0 = sin enfermedad |

Se usan los tipos **anulables** de pandas (`Int16`, `Float64`, `boolean`) en lugar de los
de NumPy porque estos datos tienen nulos en las 14 columnas: un `int64` de NumPy no
admite `NaN` y forzaría a degradar los enteros a `float`.

## 👷 Data preparation

### Definición del dominio válido

La estrategia para unificar los nulos es **validar contra el dominio documentado**: todo
valor que no pertenezca al dominio de su columna es un nulo disfrazado y se convierte en
`pd.NA`. Es más defendible que ir tapando casos uno a uno, y de paso deja escritas las
primeras reglas de validación de datos que el EDA (issue #10) va a reutilizar.

Los rangos numéricos son deliberadamente **amplios** (límites de plausibilidad clínica,
no percentiles observados) para no descartar ningún dato real.

In [ ]:
# Dominio válido de cada columna, según data/01_raw/datos_corazon_Info.txt
CATEGORIAS_VALIDAS = {
    "sex": ["Female", "Male"],
    "chest_pain": ["asymptomatic", "nonanginal", "nontypical", "typical"],
    "rest_ecg": ["ST-T wave abnormality", "left ventricular hypertrophy", "normal"],
    "thal": ["fixed", "normal", "reversable"],
}

# Rangos de plausibilidad clínica para las variables continuas
RANGOS_NUMERICOS = {
    "age": (0, 120),
    "rest_bp": (50, 250),
    "chol": (100, 600),
    "max_hr": (60, 220),
    "old_peak": (0.0, 10.0),
}

# Variables numéricas con un conjunto cerrado de valores admitidos
VALORES_DISCRETOS = {
    "fbs": [0, 1],
    "exang": [0, 1],
    "disease": [0, 1],
    "ca": [0, 1, 2, 3],
    "slope": [1, 2, 3],
}

COLUMNAS_ENTERAS = ["age", "rest_bp", "chol", "max_hr"]
COLUMNAS_DECIMALES = ["old_peak"]
COLUMNAS_BOOLEANAS = ["fbs", "exang", "disease"]

### Paso 1: normalizar el texto

Antes de validar hay que quitar los espacios sobrantes, o categorías legítimas como
`'left ventricular hypertrophy '` (con espacio final) se descartarían por no coincidir
exactamente con su dominio.

In [ ]:
corazon = corazon_raw.copy()

columnas_texto = corazon.select_dtypes(include="str").columns
espacios_antes = sum(
    (corazon[col].dropna() != corazon[col].dropna().str.strip()).sum() for col in columnas_texto
)

for col in columnas_texto:
    # Se quitan espacios sobrantes y las cadenas vacías se vuelven nulos
    corazon[col] = corazon[col].str.strip().replace("", None)

print(f"Celdas con espacios sobrantes corregidas: {espacios_antes}")
print(f"Categorias de rest_ecg tras normalizar: {sorted(corazon['rest_ecg'].dropna().unique())}")

Celdas con espacios sobrantes corregidas: 1387
Categorias de rest_ecg tras normalizar: ['3563', '36653', '435647', '5653', '5678', 'ST-T wave abnormality', 'left ventricular hypertrophy', 'normal']


### Paso 2: identificar los valores fuera de dominio

Antes de convertirlos, se cuantifica cuántos valores no pertenecen al dominio de su
columna. Estos son los **nulos disfrazados**.

In [ ]:
def valores_fuera_de_dominio(serie: pd.Series, columna: str) -> pd.Series:
    """Devuelve los valores no nulos que no pertenecen al dominio de la columna."""
    presentes = serie.dropna()
    if columna in CATEGORIAS_VALIDAS:
        return presentes[~presentes.isin(CATEGORIAS_VALIDAS[columna])]

    numerica = pd.to_numeric(presentes, errors="coerce")
    if columna in VALORES_DISCRETOS:
        invalido = ~numerica.isin(VALORES_DISCRETOS[columna])
    else:
        minimo, maximo = RANGOS_NUMERICOS[columna]
        invalido = numerica.isna() | ~numerica.between(minimo, maximo)
    return presentes[invalido]


diagnostico = [
    {
        "columna": col,
        "fuera_de_dominio": valores_fuera_de_dominio(corazon[col], col).size,
        "ejemplos": ", ".join(
            sorted(valores_fuera_de_dominio(corazon[col], col).astype(str).unique())[:4]
        ),
    }
    for col in corazon.columns
]

diagnostico_df = pd.DataFrame(diagnostico).set_index("columna")
print(f"Total de valores fuera de dominio: {diagnostico_df['fuera_de_dominio'].sum()}")
diagnostico_df

Total de valores fuera de dominio: 33


,fuera_de_dominio,ejemplos
columna,,
age,2,"fggfds, sdg"
sex,3,"2345, 45, 765"
chest_pain,3,"2345, 2435, 3456"
rest_bp,2,"fsgh, wety"
chol,2,"sfdywe, wtey"
fbs,0,
rest_ecg,5,"3563, 36653, 435647, 5653"
max_hr,1,adfs
exang,2,"adfs, f"


Los valores fuera de dominio son exactamente el ruido catalogado en el issue #7: texto
basura en columnas numéricas y números sin sentido en categóricas. **Ningún valor
legítimo cayó fuera de los rangos de plausibilidad**, lo que confirma que el ruido es
puramente sintético y que no hay valores clínicamente absurdos que corregir.

### Paso 3: unificar nulos y convertir a los tipos correctos

Cada conversión hace las dos cosas a la vez: lo que no pertenece al dominio se vuelve
`pd.NA` y lo que sí pertenece queda con su tipo definitivo.

In [ ]:
# Enteros anulables
for col in COLUMNAS_ENTERAS:
    numerica = pd.to_numeric(corazon[col], errors="coerce")
    minimo, maximo = RANGOS_NUMERICOS[col]
    corazon[col] = numerica.where(numerica.between(minimo, maximo)).astype("Int16")

# Decimales anulables
for col in COLUMNAS_DECIMALES:
    numerica = pd.to_numeric(corazon[col], errors="coerce")
    minimo, maximo = RANGOS_NUMERICOS[col]
    corazon[col] = numerica.where(numerica.between(minimo, maximo)).astype("Float64")

# Booleanos anulables (1 = True, 0 = False)
for col in COLUMNAS_BOOLEANAS:
    numerica = pd.to_numeric(corazon[col], errors="coerce")
    corazon[col] = numerica.where(numerica.isin(VALORES_DISCRETOS[col])).astype("boolean")

# Conteo de vasos (0-3) y pendiente del segmento ST (ordinal 1 < 2 < 3)
for col in ["ca", "slope"]:
    numerica = pd.to_numeric(corazon[col], errors="coerce")
    corazon[col] = numerica.where(numerica.isin(VALORES_DISCRETOS[col])).astype("Int8")

# Categóricas nominales
for col, categorias in CATEGORIAS_VALIDAS.items():
    corazon[col] = pd.Categorical(
        corazon[col].where(corazon[col].isin(categorias)), categories=categorias
    )

corazon.info()

<class 'pandas.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   age         2998 non-null   Int16   
 1   sex         2966 non-null   category
 2   chest_pain  2944 non-null   category
 3   rest_bp     2947 non-null   Int16   
 4   chol        2943 non-null   Int16   
 5   fbs         2933 non-null   boolean 
 6   rest_ecg    2832 non-null   category
 7   max_hr      2858 non-null   Int16   
 8   exang       2877 non-null   boolean 
 9   old_peak    2878 non-null   Float64 
 10  slope       2878 non-null   Int8    
 11  ca          2867 non-null   Int8    
 12  thal        2900 non-null   category
 13  disease     2919 non-null   boolean 
dtypes: Float64(1), Int16(4), Int8(2), boolean(3), category(4)
memory usage: 104.4 KB


### Verificación de la conversión

Se compara el estado antes y después: cuántos nulos había en el RAW, cuántos se
recuperaron al desenmascarar el ruido y cuál es el tipo final de cada columna.

In [ ]:
verificacion = pd.DataFrame(
    {
        "dtype_raw": corazon_raw.dtypes.astype(str),
        "dtype_final": corazon.dtypes.astype(str),
        "nulos_raw": corazon_raw.isna().sum(),
        "nulos_final": corazon.isna().sum(),
    }
)
verificacion["nulos_desenmascarados"] = verificacion["nulos_final"] - verificacion["nulos_raw"]
verificacion["pct_nulos_final"] = (corazon.isna().mean() * 100).round(2)
verificacion

,dtype_raw,dtype_final,nulos_raw,nulos_final,nulos_desenmascarados,pct_nulos_final
age,str,Int16,30,32,2,1.06
sex,str,category,61,64,3,2.11
chest_pain,str,category,83,86,3,2.84
rest_bp,str,Int16,81,83,2,2.74
chol,str,Int16,85,87,2,2.87
fbs,float64,boolean,97,97,0,3.20
rest_ecg,str,category,193,198,5,6.53
max_hr,str,Int16,171,172,1,5.68
exang,str,boolean,151,153,2,5.05
old_peak,str,Float64,150,152,2,5.02


In [ ]:
# El numero de filas no cambia: esta capa solo tipa, no filtra
assert len(corazon) == len(corazon_raw), "La capa intermediate no debe eliminar filas"

nulos_nuevos = int((corazon.isna().sum() - corazon_raw.isna().sum()).sum())
print(f"Filas: {len(corazon_raw)} -> {len(corazon)} (sin cambios)")
print(f"Valores desenmascarados como nulos: {nulos_nuevos}")
print(f"Columnas de texto sin tipar restantes: {len(corazon.select_dtypes(include='str').columns)}")

Filas: 3030 -> 3030 (sin cambios)
Valores desenmascarados como nulos: 33
Columnas de texto sin tipar restantes: 0


### La descripción estadística ahora sí es posible

Con los tipos corregidos, `describe()` produce por fin información útil. Es la mejor
prueba de que el saneamiento funcionó.

In [ ]:
corazon.describe().T

,count,mean,std,min,25%,50%,75%,max
age,2998.0,54.462308,9.022369,29.0,48.0,56.0,61.0,77.0
rest_bp,2947.0,131.619952,17.571968,94.0,120.0,130.0,140.0,200.0
chol,2943.0,246.722732,51.596773,126.0,211.0,241.0,275.0,564.0
max_hr,2858.0,149.558083,22.876201,71.0,133.0,153.0,166.0,202.0
old_peak,2878.0,1.03975,1.162667,0.0,0.0,0.8,1.6,6.2
slope,2878.0,1.600069,0.616842,1.0,1.0,2.0,2.0,3.0
ca,2867.0,0.674573,0.936836,0.0,0.0,0.0,1.0,3.0


In [ ]:
# Distribución de las variables categóricas, booleanas y discretas
for col in [*CATEGORIAS_VALIDAS, *COLUMNAS_BOOLEANAS, "ca", "slope"]:
    print(f"--- {col} ---")
    print(corazon[col].value_counts(dropna=False).to_string())
    print()

--- sex ---


sex
Male      2017
Female     949
NaN         64

--- chest_pain ---
chest_pain
asymptomatic    1394
nonanginal       840
nontypical       486
typical          224
NaN               86

--- rest_ecg ---
rest_ecg
normal                          1409
left ventricular hypertrophy    1387
NaN                              198
ST-T wave abnormality             36

--- thal ---
thal
normal        1599
reversable    1129
fixed          172
NaN            130

--- fbs ---
fbs
False    2498
True      435
<NA>       97

--- exang ---
exang
False    1929
True      948
<NA>      153

--- disease ---
disease
False    1576
True     1343
<NA>      111

--- ca ---
ca
0       1684
1        624
2        367
3        192
<NA>     163

--- slope ---
slope
1       1353
2       1323
3        202
<NA>     152



## 💾 Almacenamiento en parquet

El dataset tipado se guarda en `data/02_intermediate/corazon.parquet`. Se elige parquet
sobre CSV porque **preserva los dtypes**: un CSV volvería a entregar todo como texto en la
siguiente etapa y habría que repetir esta conversión.

In [ ]:
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
ruta_parquet = INTERMEDIATE_DIR / "corazon.parquet"

corazon.to_parquet(ruta_parquet, index=False)

tamano_csv = (RAW_DIR / "corazon.csv").stat().st_size / 1024
tamano_parquet = ruta_parquet.stat().st_size / 1024
print(f"Guardado en: {ruta_parquet.relative_to(project_root)}")
print(f"CSV RAW:  {tamano_csv:8.1f} KB")
print(f"Parquet:  {tamano_parquet:8.1f} KB ({tamano_parquet / tamano_csv:.1%} del CSV)")

Guardado en: data\02_intermediate\corazon.parquet
CSV RAW:     217.9 KB
Parquet:      19.2 KB (8.8% del CSV)


In [ ]:
# Verificacion de round-trip: releer el parquet debe devolver exactamente lo mismo
corazon_releido = pd.read_parquet(ruta_parquet)

assert corazon_releido.equals(corazon), "El parquet no reprodujo los datos"
assert corazon_releido.dtypes.equals(corazon.dtypes), "El parquet no preservo los dtypes"

print("Round-trip verificado: valores y dtypes identicos tras releer el parquet")
corazon_releido.dtypes.to_frame("dtype")

Round-trip verificado: valores y dtypes identicos tras releer el parquet


,dtype
age,Int16
sex,category
chest_pain,category
rest_bp,Int16
chol,Int16
fbs,boolean
rest_ecg,category
max_hr,Int16
exang,boolean
old_peak,Float64


## 📊 Analysis of Results and Conclusions

El dataset RAW quedó convertido en un dataset **tipado y con nulos unificados**, guardado
en `data/02_intermediate/corazon.parquet`.

1. **El problema real del CSV era de tipos, no de contenido.** Trece de catorce columnas
   llegaban como texto por culpa de 33 celdas de ruido sobre 42.420. Menos del 0.1% de
   valores corruptos bastaba para inutilizar la descripción estadística del 100% del
   dataset.

2. **Todo el ruido resultó ser nulos disfrazados.** Validando contra el dominio
   documentado, los valores fuera de rango se desenmascararon como nulos: textos basura
   en columnas numéricas y números sin sentido en categóricas. No hubo que *corregir*
   ningún valor, solo reconocerlo como ausente.

3. **No hay valores clínicamente absurdos.** Los rangos de plausibilidad (`age` 0-120,
   `rest_bp` 50-250, `chol` 100-600, `max_hr` 60-220, `old_peak` 0.0-10.0) no excluyeron
   ni un solo dato real: los valores observados caen todos dentro. El ruido es puramente
   sintético y localizable, no un problema de captura de datos.

4. **Los espacios sobrantes eran la trampa más peligrosa, y no por poco.** Las **1387
   celdas** de `rest_ecg` con el valor `'left ventricular hypertrophy '` traían un espacio
   final: el 45.8% de la columna. Si se hubiera validado contra el dominio *antes* de
   normalizar el texto, esas 1387 filas no habrían coincidido con ninguna categoría
   válida y se habrían convertido en nulos, dejando `rest_ecg` con 1585 ausentes (52.3%)
   y probablemente inservible. El orden de los pasos —normalizar primero, validar
   después— importa más que cualquiera de los dos por separado.

5. **Se necesitan los tipos anulables de pandas.** Como las 14 columnas tienen nulos, los
   tipos de NumPy no sirven: un `int64` no admite valores ausentes y habría degradado
   todos los enteros a `float`. `Int16`/`Int8`/`Float64`/`boolean` conservan la semántica.

6. **Parquet no preserva las categóricas de enteros.** Al probar `slope` como
   `Categorical` ordenado con categorías `[1, 2, 3]`, el round-trip lo devolvía como
   `float64`, perdiendo tanto el tipo como el orden. Por eso `slope` quedó como `Int8`
   con su naturaleza ordinal documentada: el orden entero ya codifica la ordinalidad y
   sobrevive al guardado. La verificación de round-trip fue la que detectó esto.

7. **La capa conserva las 3030 filas.** El dataset intermedio es el RAW tipado, no el RAW
   limpio. Nada se dedujo, filtró ni imputó, de forma que sigue siendo trazable fila a
   fila contra el CSV original.

## 💡 Proposals and Ideas

1. **Siguiente etapa (issue #10, EDA)**: partir de `data/02_intermediate/corazon.parquet`,
   no del CSV. Los tipos ya están resueltos y el EDA puede concentrarse en
   distribuciones, correlaciones y outliers.
2. **Deduplicar en la capa `03_primary`, no aquí.** Las 2462 filas duplicadas (81.3%)
   siguen presentes a propósito. Al deduplicar hay que hacerlo **antes** de cualquier
   `train_test_split`, o el mismo paciente cae en train y test e infla las métricas.
3. **Formalizar las reglas de validación.** Los diccionarios `CATEGORIAS_VALIDAS`,
   `RANGOS_NUMERICOS` y `VALORES_DISCRETOS` son ya un contrato de datos; conviene
   moverlos a `src/data/` para reutilizarlos desde el pipeline y desde los tests, en vez
   de reescribirlos en cada notebook.
4. **Decidir el tratamiento de los nulos en el issue #11**, no antes: la imputación
   depende de la distribución de cada variable, que es justamente lo que el EDA va a
   caracterizar. La imputación debe ir dentro del pipeline de scikit-learn para que se
   ajuste solo con los datos de train.
5. **No imputar el target.** Las filas sin `disease` válido deben descartarse en la etapa
   de limpieza; imputar la variable objetivo inventaría supervisión.
6. **Revisar `ca` y `slope` en el encoding.** Quedaron como enteros por conveniencia de
   almacenamiento, pero ninguna es continua: `ca` es un conteo y `slope` es ordinal.
   Conviene decidir explícitamente en el issue #11 si van como ordinales o como
   categóricas one-hot.

## 📖 References

- Diccionario de variables y nota del curso: `data/01_raw/datos_corazon_Info.txt`.
- Metodología de la etapa: [Jose R. Zapata - Proyecto de ciencia de datos: 2. Exploration](https://joserzapata.github.io/post/ciencia-datos-proyecto-python/2-exploration/).
- Convención de capas de datos del proyecto: `data/README.md`
  ([data engineering convention de Kedro](https://docs.kedro.org/en/stable/faq/faq.html#what-is-data-engineering-convention)).
- [Tipos de datos anulables en pandas (nullable integer / boolean)](https://pandas.pydata.org/docs/user_guide/integer_na.html).
- [Datos categóricos en pandas](https://pandas.pydata.org/docs/user_guide/categorical.html).
- [Formato Apache Parquet](https://parquet.apache.org/docs/).
- Notebook anterior: `notebooks/1-data/01-jrdt-descarga_datos-2026_08_15.ipynb` (issue #7).